In [1]:
# Load required packages
using DelimitedFiles
using LinearAlgebra
using Revise

# Add JPEC to path and load
push!(LOAD_PATH, joinpath(@__DIR__, "../.."))
using JPEC

# Initialize globals with proper grid parameters
# The kernel function expects coordinates on a regular theta grid
mth = 512  # Number of poloidal grid points
globals = JPEC.VacuumMod.VacuumGlobalsType(
    mth = mth,
    mth1 = mth + 1,
    mth2 = mth + 2,
    dth = 2π / mth  # Critical: theta grid spacing
)
settings = JPEC.VacuumMod.VacuumSettingsType()

JPEC.VacuumMod.VacuumSettingsType(JPEC.VacuumMod.Modes(480, [0, 0, 0, 0, 0, 0, 0, 1, 0], true, 1), JPEC.VacuumMod.Vacdat(6, 0.05, 1.5, 0.0, 0.5, 0.05, 500, 1.0e-5, 37, 6, 0, 0, 15.01, 0.001, 1), JPEC.VacuumMod.Shape(0, 0, 0, 100.0, 1.0, 20.0, 170.0, 1.0, 0.0, 1.0, 0.932, 17.0, 0.02, 2.5, 1.0, 0.0), JPEC.VacuumMod.Diagns(false, 0, 0, 1, 128, 0, 3, 1, 0.0, 32, 0.01, 1.6, 0.5, 1.0, 0.001, 21, 21, 0, 6, 11, 0.02, 0.7, 2.7, -1.5, 1.5, 2), JPEC.VacuumMod.Sprk(0, 0, 16, 0, 0.0, 1, 1, 1, 1, 1, -1, -1, -1, 1, 1, 0, 1.6, [1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6  …  1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6]), false)

In [2]:
# Simple test with properly sized coordinate arrays on theta grid
# The kernel expects coordinates that match the theta grid size (mth+1 points)

mth_test = 512
npts = mth_test + 1  # Need mth+1 points for theta grid [0, 2π]

# Create simple circular test geometry
theta_test = LinRange(0, 2π, npts)
R0 = 2.0  # Major radius
a = 0.5   # Minor radius

# Observer points (plasma boundary)
test_xobs = R0 .+ a .* cos.(theta_test)
test_zobs = a .* sin.(theta_test)

# Source points (slightly inside plasma)
test_xsce = R0 .+ 0.9*a .* cos.(theta_test)
test_zsce = 0.9*a .* sin.(theta_test)

# Initialize small test matrices
test_grdgre = zeros(Float64, npts, npts)
test_gren = zeros(Float64, npts, npts)

println("Test array sizes:")
println("  xobs: $(length(test_xobs)), zobs: $(length(test_zobs))")
println("  xsce: $(length(test_xsce)), zsce: $(length(test_zsce))")
println("  Output matrices: $(size(test_grdgre))")
println("  globals.dth = $(globals.dth)")
println("  globals.mth = $(globals.mth)")

try
    println("\nCalling kernel!...")
    JPEC.VacuumMod.kernel!(
        test_grdgre, test_gren, 
        test_xobs, test_zobs, 
        test_xsce, test_zsce, 
        1, 1, -1, 1, 1, false, globals, settings
    )
    
    println("✓ Kernel executed!")
    println("  grdgre sum: $(sum(test_grdgre))")
    println("  gren sum: $(sum(test_gren))")
    println("  Non-zero elements in grdgre: $(count(!iszero, test_grdgre))")
    println("  Non-zero elements in gren: $(count(!iszero, test_gren))")
catch e
    println("✗ Error: $e")
    showerror(stdout, e, catch_backtrace())
end

Test array sizes:
  xobs: 513, zobs: 513
  xsce: 513, zsce: 513
  Output matrices: (513, 513)
  globals.dth = 0.01227184630308513
  globals.mth = 512

Calling kernel!...
✓ Kernel executed!
  grdgre sum: -983.9503967064858
  gren sum: 784.3476576663111
  Non-zero elements in grdgre: 262144
  Non-zero elements in gren: 262144


# TRY THIS STUFF AFTER THE TEST CASE ABOVE RUNS THROUGH -- to compare with Fortran outputs

## Step 1: Load and Parse Test Data

First, let's read the data from output_data.txt. The file has space-separated numbers that we need to parse into arrays.

# Kernel Function Test for Vacuum_vac.jl

This notebook tests the `kernel!` function from `Vacuum_vac.jl` using test data from `output_data.txt`.

## Test Setup

The output_data.txt file contains space-separated numerical data that appears to be organized in rows. We'll:
1. Load the JPEC module
2. Parse the input data from output_data.txt
3. Set up the test parameters for the kernel function
4. Run the kernel function and compare results

In [2]:
# Read the output_data.txt file
data_file = joinpath(@__DIR__, "output_data.txt")

println("Reading file: $data_file")
println("File exists: $(isfile(data_file))")
println()

# Read all lines
lines = readlines(data_file)
println("Total lines in file: $(length(lines))")

# Helper function to parse a line of comma-separated numbers
function parse_number_line(line)
    tokens = split(line, ',')
    return [parse(Float64, strip(token)) for token in tokens if !isempty(strip(token))]
end

# Parse the five lines
if length(lines) < 5
    error("Expected 5 lines in output_data.txt, but found only $(length(lines)) lines!")
end

println("\nParsing data from first 5 lines:")
xobs = parse_number_line(lines[1])
println("  Line 1 (xobs): $(length(xobs)) values")

zobs = parse_number_line(lines[2])
println("  Line 2 (zobs): $(length(zobs)) values")

xsce = parse_number_line(lines[3])
println("  Line 3 (xsce): $(length(xsce)) values")

zsce = parse_number_line(lines[4])
println("  Line 4 (zsce): $(length(zsce)) values")

params = parse_number_line(lines[5])
println("  Line 5 (params): $(length(params)) values")

println("\n✓ Successfully parsed all data!")
println("\nData summary:")
println("  xobs: $(length(xobs)) points, range [$(minimum(xobs)), $(maximum(xobs))]")
println("  zobs: $(length(zobs)) points, range [$(minimum(zobs)), $(maximum(zobs))]")
println("  xsce: $(length(xsce)) points, range [$(minimum(xsce)), $(maximum(xsce))]")
println("  zsce: $(length(zsce)) points, range [$(minimum(zsce)), $(maximum(zsce))]")
println("  params: $(params)")

# Extract kernel function parameters
j1_input = Int(params[1])
j2_input = Int(params[2])
isgn_input = Int(params[3])
iopw_input = Int(params[4])
iops_input = Int(params[5])
wall_flag_input = params[6] != 0  # Convert to boolean

println("\nKernel parameters:")
println("  j1 = $j1_input")
println("  j2 = $j2_input")
println("  isgn = $isgn_input")
println("  iopw = $iopw_input")
println("  iops = $iops_input")
println("  wall_flag = $wall_flag_input")

Reading file: /Users/priyansh/Documents/Git/JPEC/notebooks/vacuum_tests/output_data.txt
File exists: true

Total lines in file: 5

Parsing data from first 5 lines:
  Line 1 (xobs): 517 values
  Line 2 (zobs): 517 values
  Line 3 (xsce): 517 values
  Line 4 (zsce): 517 values
  Line 5 (params): 6 values

✓ Successfully parsed all data!

Data summary:
  xobs: 517 points, range [0.0, 2.2891936601349068]
  zobs: 517 points, range [-1.1134694034127417, 0.9616419248701459]
  xsce: 517 points, range [0.0, 2.2891936601349068]
  zsce: 517 points, range [-1.1134694034127417, 0.9616419248701459]
  params: [1.0, 1.0, -1.0, 1.0, 1.0, 0.0]

Kernel parameters:
  j1 = 1
  j2 = 1
  isgn = -1
  iopw = 1
  iops = 1
  wall_flag = false


## Step 2: Set Up Kernel Function Parameters

Data has been loaded and kernel parameters extracted from the input file.

In [3]:
# Set up parameters for kernel function test
nobs = length(xobs)
nsrc = length(xsce)

# Validate data consistency
if length(xobs) != length(zobs)
    error("xobs and zobs have different lengths: $(length(xobs)) vs $(length(zobs))")
end
if length(xsce) != length(zsce)
    error("xsce and zsce have different lengths: $(length(xsce)) vs $(length(zsce))")
end

println("Data validation passed!")
println()

# Initialize output matrices
grdgre = zeros(Float64, nobs, nsrc)
gren = zeros(Float64, nobs, nsrc)

# Use parameters loaded from input file
j1 = j1_input
j2 = j2_input
isgn = isgn_input
iopw = iopw_input
iops = iops_input
wall_flag = wall_flag_input

println("Test configuration:")
println("  Observer points: $nobs")
println("  Source points: $nsrc")
println("  Observer type (j1): $j1 (1=plasma, 2=wall)")
println("  Source type (j2): $j2 (1=plasma, 2=wall)")
println("  Sign parameter (isgn): $isgn")
println("  Wall option (iopw): $iopw")
println("  Log singularity option (iops): $iops")
println("  Wall flag: $wall_flag")
println()
println("Output matrix dimensions:")
println("  grdgre: $(size(grdgre))")
println("  gren: $(size(gren))")

Data validation passed!

Test configuration:
  Observer points: 517
  Source points: 517
  Observer type (j1): 1 (1=plasma, 2=wall)
  Source type (j2): 1 (1=plasma, 2=wall)
  Sign parameter (isgn): -1
  Wall option (iopw): 1
  Log singularity option (iops): 1
  Wall flag: false

Output matrix dimensions:
  grdgre: (517, 517)
  gren: (517, 517)


## Step 3: Run the Kernel Function

The `kernel!` function signature from Vacuum_vac.jl:
```julia
kernel!(grdgre, gren, xobs, zobs, xsce, zsce, j1, j2, isgn, iopw, iops, wall_flag)
```

Now we'll call the kernel function with our test data.

## Step 4: Analyze Results

Visualize and analyze the kernel function output.

In [4]:
# Run the kernel function
# Note: This may fail if additional setup/initialization is required
try
    JPEC.VacuumMod.kernel!(
        grdgre, gren, 
        xobs, zobs, 
        xsce, zsce, 
        j1, j2, isgn, iopw, iops, wall_flag, globals
    )
    
    println("✓ Kernel function executed successfully!")
    println()
    println("Result statistics:")
    println("  grdgre: min=$(minimum(grdgre)), max=$(maximum(grdgre)), mean=$(sum(grdgre)/length(grdgre))")
    println("  gren: min=$(minimum(gren)), max=$(maximum(gren)), mean=$(sum(gren)/length(gren))")
    
catch e
    println("✗ Error running kernel function:")
    println(e)
    println()
    println("Stack trace:")
    for (exc, bt) in Base.catch_stack()
        showerror(stdout, exc, bt)
        println()
    end
    println()
end

✗ Error running kernel function:
MethodError(JPEC.VacuumMod.kernel!, ([0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0

Excessive output truncated after 524290 bytes.

 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0

InterruptException: InterruptException:

In [15]:
# Debug: Check what's in the result matrices
println("===== RESULT DIAGNOSTICS =====")
println("grdgre exists: $(isdefined(Main, :grdgre))")
println("gren exists: $(isdefined(Main, :gren))")
println()

if isdefined(Main, :grdgre) && isdefined(Main, :gren)
    println("grdgre size: $(size(grdgre))")
    println("gren size: $(size(gren))")
    println()
    
    println("grdgre all zeros: $(all(grdgre .== 0))")
    println("gren all zeros: $(all(gren .== 0))")
    println()
    
    println("grdgre sum: $(sum(grdgre))")
    println("gren sum: $(sum(gren))")
    println()
    
    println("grdgre first 3x3:")
    display(grdgre[1:min(3,size(grdgre,1)), 1:min(3,size(grdgre,2))])
    println()
    
    println("gren first 3x3:")
    display(gren[1:min(3,size(gren,1)), 1:min(3,size(gren,2))])
else
    println("ERROR: Result matrices not defined!")
    println("Make sure to run the kernel execution cell first.")
end

3×3 Matrix{Float64}:
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

3×3 Matrix{Float64}:
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0